# Biomarker hits — Kaplan-Meier curves by marker status (weighted vs unweighted)

Renders **every** FDR-significant `marker x ICI` interaction hit from
[01_pipeline.ipynb](01_pipeline.ipynb) to a PNG, each hit drawn
twice in one figure: **IPTW-weighted** (`ATE`) beside **unweighted** (`noIPTW`), the same two
weightings `run_IPTW_analysis` fits.

Output goes to `KM_OUT_DIR` (see the export cell), one subdirectory per cohort/ps_model
specification, plus a manifest CSV recording every hit's outcome — written, skipped and why,
with group counts and ESS. Run top to bottom; nothing needs editing for a full export.

### What is plotted
Four curves per panel — the 2x2 of marker status x ICI exposure:

| | marker− | marker+ |
|---|---|---|
| **non-ICI** | reference | marker's prognostic effect off ICI |
| **ICI** | ICI effect at marker− | |

The interaction HR the screen reports is *not* a contrast between two of these curves; it is
the ratio of the two marker hazard ratios (marker+ vs marker− within ICI, over the same within
non-ICI). The KM panel shows whether that ratio is coming from a real separation in the curves
or from one thin arm — which a HR and a q-value alone cannot tell you.

### Population: reconstructed to match the Cox fit
A KM drawn on the raw `IPTW_df_*` parquet is **not** the population the hit was estimated on.
`run_IPTW_analysis` puts each cancer type through three steps first, and this notebook calls
that module's own functions to repeat them rather than reimplementing them:

1. **PS recalibration** within the cancer-type subset (`recalibrate_propensity_within_subset`) —
   cancer-type screens only; `pan_cancer` keeps the original propensity score.
2. **Common-support trimming** to the overlapping ${0.5, 99.5}$ propensity percentiles of the
   two arms — this drops patients.
3. **Stabilized ATE weights** $w = p/e$ (treated), $(1-p)/(1-e)$ (control), truncated at the
   1st/99th percentiles.

The weighted curves use those weights; the unweighted curves use the same trimmed rows with
weight 1. So the two panels differ **only** in the weighting, not in who is in them — which is
the comparison worth looking at. `RESTRICT_TO_TRIMMED = False` opts out and draws the untrimmed
frame instead, for a look at what trimming removed.

### Weighted KM
`lifelines`' `KaplanMeierFitter` accepts `weights=`, giving the IPTW-adjusted survival estimate.
Confidence bands on a weighted KM are anti-conservative (they treat weights as frequencies and
so overstate the effective sample size), so weighted panels report **ESS** beside N and the
bands are off by default — see `SHOW_CI`.

**Reads only.** Nothing here writes to the pipeline's data directories — the only writes are
the PNGs and the manifest under `KM_OUT_DIR`.

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.exceptions import StatisticalWarning
from sklearn.linear_model import LogisticRegression

import config
from shared.polars_utils import filter_finite_rows, finite_or_zero

# Trimming/weighting constants come from the analysis module itself where it is
# importable, so a change there cannot leave this notebook drawing a
# differently-trimmed population than the HR it prints above the curves.
#
# The import is guarded because `run_IPTW_analysis` pulls in the whole screening
# stack (statsmodels, joblib, tqdm) for machinery this notebook never runs -- FDR
# correction and the parallel marker screen. On an environment without those, the
# fallbacks below keep the notebook usable; they are asserted equal to the
# module's values whenever the import does succeed, so the two cannot drift
# silently.
COMMON_SUPPORT_PCT = (0.5, 99.5)
IPTW_TRUNC_PCT = (1, 99)

try:
    from pipelines.biomarkers import run_IPTW_analysis as _screen
    assert tuple(_screen.COMMON_SUPPORT_PCT) == COMMON_SUPPORT_PCT, (
        f"COMMON_SUPPORT_PCT drifted: notebook {COMMON_SUPPORT_PCT} vs "
        f"run_IPTW_analysis {tuple(_screen.COMMON_SUPPORT_PCT)}")
    assert tuple(_screen.IPTW_TRUNC_PCT) == IPTW_TRUNC_PCT, (
        f"IPTW_TRUNC_PCT drifted: notebook {IPTW_TRUNC_PCT} vs "
        f"run_IPTW_analysis {tuple(_screen.IPTW_TRUNC_PCT)}")
    COMMON_SUPPORT_PCT = tuple(_screen.COMMON_SUPPORT_PCT)
    IPTW_TRUNC_PCT = tuple(_screen.IPTW_TRUNC_PCT)
    _SCREEN_IMPORTED = True
except ImportError as _exc:
    _SCREEN_IMPORTED = False
    print(f"[note] run_IPTW_analysis not importable ({_exc}); using the trimming "
          f"constants restated in this cell. Verify they still match the script.")


def recalibrate_propensity_within_subset(df, ps_col="ICI_prediction", treat_col="PX_on_ICI"):
    """Refit the propensity score within a cancer-type subset.

    Mirrors the function of the same name in `run_IPTW_analysis`; kept here so the
    notebook does not need that module's screening dependencies. The equality
    check below runs whenever the real one is importable.
    """
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(df.select(ps_col).to_numpy(), df[treat_col].to_numpy().astype(int))
    return lr.predict_proba(df.select(ps_col).to_numpy())[:, 1]

BIOMARKER_PATH = config.BIOMARKER_PATH
COMPILED_DIR = os.path.join(BIOMARKER_PATH, "compiled_results/")
HITS_FILE = os.path.join(COMPILED_DIR, "track2_all_significant_hits.csv")

# --- Display options ---
TIME_UNIT = "months"        # "months" | "days"
TIME_DIVISOR = 30.44 if TIME_UNIT == "months" else 1.0
MAX_FOLLOWUP = 60           # x-axis cap in TIME_UNIT; None = full follow-up
SHOW_CI = False             # CI bands. On a weighted KM these are anti-conservative
                            # (weights are treated as frequencies), so they are off
                            # by default and the weighted panel reports ESS instead.
SHOW_AT_RISK = True         # at-risk counts under each panel
RESTRICT_TO_TRIMMED = True  # reproduce the analysis population (recalibrate + trim).
                            # False = draw the untrimmed cancer-type frame.

# Four-way palette: colour = marker status, linestyle = ICI exposure.
GROUP_STYLE = {
    (0, 0): dict(color="#4C72B0", linestyle="--", label="marker- / non-ICI"),
    (1, 0): dict(color="#C44E52", linestyle="--", label="marker+ / non-ICI"),
    (0, 1): dict(color="#4C72B0", linestyle="-",  label="marker- / ICI"),
    (1, 1): dict(color="#C44E52", linestyle="-",  label="marker+ / ICI"),
}

print(f"repo root:   {REPO_ROOT}")
print(f"Python:    {sys.executable}")
print(f"Compiled:  {COMPILED_DIR}")
print(f"Hits file: {HITS_FILE}")
print(f"           {'found' if os.path.exists(HITS_FILE) else 'MISSING -- run stage 6 (compile_IPTW_results) first'}")

## The hits

`track2_all_significant_hits.csv` — every FDR-significant interaction, one row per
(marker, cancer type, cohort, ps_model, weight_type). A marker significant under both
weightings appears twice; the KM panel below draws both weightings regardless of which
one earned the hit, so those rows collapse to one figure.

In [ ]:
if not os.path.exists(HITS_FILE):
    raise FileNotFoundError(
        f"{HITS_FILE} not found. Run stage 6 (compile_IPTW_results) in "
        "01_pipeline.ipynb first."
    )

try:
    hits = pl.read_csv(HITS_FILE)
except pl.exceptions.NoDataError:
    hits = pl.DataFrame()

if hits.is_empty():
    print("Compiled hits file is empty -- the screen found no FDR-significant interactions.")
else:
    print(f"{hits.height} significant hit rows "
          f"({hits.select(['marker', 'cancer_type']).unique().height} unique marker x cancer type)\n")
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        print(hits.group_by(["cohort", "ps_model", "weight_type"])
                  .agg(pl.len().alias("n_hits"),
                       pl.col("cancer_type").n_unique().alias("n_cancer_types"))
                  .sort(["cohort", "ps_model", "weight_type"]))

In [ ]:
# One row per (marker, cancer_type, cohort, ps_model) -- the unit a KM panel is drawn for.
# `weight_type` is dropped from the key because the panel shows both weightings; the
# hit's own weighting is kept as `sig_in` so a panel says which screen flagged it.
HIT_KEY = ["marker", "cancer_type", "cohort", "ps_model"]

if hits.is_empty():
    panels = pl.DataFrame()
else:
    panels = (hits
              .with_columns(pl.col("HR_markerxICI").cast(pl.Float64, strict=False),
                            pl.col("p_markerxICI").cast(pl.Float64, strict=False))
              .group_by(HIT_KEY)
              .agg(pl.col("weight_type").unique().sort().str.join("+").alias("sig_in"),
                   pl.col("p_markerxICI").min().alias("min_p"),
                   pl.col("HR_markerxICI").first().alias("HR_first"),
                   pl.col("extreme_hr_flag").any().alias("extreme_hr")
                   if "extreme_hr_flag" in hits.columns
                   else pl.lit(False).alias("extreme_hr"))
              .sort("min_p"))
    with pl.Config(tbl_rows=40, fmt_str_lengths=60):
        print(panels)

## Reconstructing one hit's analysis population

`load_hit_population` walks the same path `run_IPTW_analysis.main()` walks for a cancer type:
filter to the type, recalibrate the propensity score within it, trim to common support, then
build stabilized truncated ATE weights. It returns the frame with `IPTW_ATE` attached plus a
dict of counts, so a panel can report how many patients the trim removed and what the effective
sample size is after weighting.

Two deliberate differences from the screen, both of which only *add* rows here:

- **No `pan_cancer` rare-type merge.** That step rewrites cancer-type dummies for the Cox design
  matrix; it changes no patient's survival, exposure, or marker, so it cannot move a KM curve.
- **No marker-support gate.** `MIN_MARKER_POS_PER_ARM` etc. decide whether a marker is *screened*.
  A compiled hit has already passed them.

In [ ]:
def _hit_label(marker: str, cancer_type: str, cohort: str, ps_model: str) -> str:
    return f"{marker} | {cancer_type} | {cohort} | {ps_model}"


def load_hit_population(marker: str, cancer_type: str, cohort: str, ps_model: str,
                        restrict_to_trimmed: bool = RESTRICT_TO_TRIMMED):
    """Rebuild the rows `run_IPTW_analysis` fit this hit on, with ATE weights.

    Returns (frame, info). `frame` carries DFCI_MRN, tt_death, death, PX_on_ICI,
    marker_value (0/1), and IPTW_ATE. `info` carries the counts a panel annotates
    with. Raises FileNotFoundError if the spec's IPTW dataset is missing, and
    ValueError if the hit's marker/cancer-type columns are not in it.
    """
    iptw_fp = os.path.join(BIOMARKER_PATH, f"IPTW_df_{cohort}_{ps_model}.parquet")
    if not os.path.exists(iptw_fp):
        raise FileNotFoundError(f"{iptw_fp} not found -- run stage 4 (generate_IPTW_df).")

    df = pl.read_parquet(iptw_fp)
    if marker not in df.columns:
        raise ValueError(f"marker {marker!r} is not a column of {os.path.basename(iptw_fp)}")

    info = {"marker": marker, "cancer_type": cancer_type, "cohort": cohort,
            "ps_model": ps_model, "n_full": df.height, "trimmed": False,
            "recalibrated": False}

    # --- Cancer-type subset (pan_cancer keeps every row, as in the screen) ---
    if cancer_type != "pan_cancer":
        ct_col = f"CANCER_TYPE_{cancer_type}"
        if ct_col not in df.columns:
            raise ValueError(f"{ct_col} is not a column of {os.path.basename(iptw_fp)}")
        df = df.filter(pl.col(ct_col).cast(pl.Boolean, strict=False))
    info["n_cancer_type"] = df.height

    # --- Model frame: same finiteness rule the fit uses, and tt_death > 0 ---
    df = filter_finite_rows(df, [marker, "PX_on_ICI", "tt_death", "death", "ICI_prediction"])
    df = df.filter(pl.col("tt_death") > 0)
    df = df.with_columns((finite_or_zero(marker) > 0).cast(pl.Int8).alias("marker_value"))
    info["n_model_frame"] = df.height

    if df.is_empty() or df["PX_on_ICI"].n_unique() < 2:
        raise ValueError(f"{_hit_label(marker, cancer_type, cohort, ps_model)}: "
                         "no rows, or only one treatment arm, before trimming.")

    if not restrict_to_trimmed:
        # Untrimmed view: weights would be built on a different population than the
        # screen used, so none are attached -- only the unweighted panel is valid.
        info["n_analysis"] = df.height
        info["n_trimmed_out"] = 0
        df = df.with_columns(pl.lit(1.0).alias("IPTW_ATE"))
        _require_four_group_support(df, marker, cancer_type, cohort, ps_model, info)
        return df, info

    # --- PS recalibration within the subset (cancer-type screens only) ---
    if cancer_type != "pan_cancer" and df["PX_on_ICI"].n_unique() >= 2 and df.height > 10:
        _check_recalibration_matches_screen(df)
        df = df.with_columns(pl.Series("ICI_prediction",
                                       recalibrate_propensity_within_subset(df)))
        info["recalibrated"] = True

    # --- Common-support trimming ---
    eps = 1e-6
    ps_raw = df["ICI_prediction"].clip(eps, 1 - eps).to_numpy()
    treat_np = df["PX_on_ICI"].to_numpy()
    ps_t, ps_c = ps_raw[treat_np == 1], ps_raw[treat_np == 0]
    lower = max(np.percentile(ps_t, COMMON_SUPPORT_PCT[0]),
                np.percentile(ps_c, COMMON_SUPPORT_PCT[0]))
    upper = min(np.percentile(ps_t, COMMON_SUPPORT_PCT[1]),
                np.percentile(ps_c, COMMON_SUPPORT_PCT[1]))
    if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
        raise ValueError(f"{_hit_label(marker, cancer_type, cohort, ps_model)}: "
                         "no propensity overlap; cannot reproduce the analysis population.")

    df = df.filter(pl.Series((ps_raw >= lower) & (ps_raw <= upper)))
    info["trimmed"] = True
    info["ps_support"] = (float(lower), float(upper))
    if df.is_empty() or df["PX_on_ICI"].n_unique() < 2:
        raise ValueError(f"{_hit_label(marker, cancer_type, cohort, ps_model)}: "
                         "no rows, or only one arm, after common-support trimming.")

    # --- Stabilized, truncated ATE weights ---
    ps = df["ICI_prediction"].clip(eps, 1 - eps).to_numpy()
    treat_mask = (df["PX_on_ICI"] == 1).to_numpy()
    p_treated = float(df["PX_on_ICI"].mean())
    w = np.where(treat_mask, p_treated / ps, (1 - p_treated) / (1 - ps))
    w = np.clip(w, *np.percentile(w, IPTW_TRUNC_PCT))
    df = df.with_columns(pl.Series("IPTW_ATE", w))

    info["n_analysis"] = df.height
    info["n_trimmed_out"] = info["n_model_frame"] - df.height
    info["ess_treated"] = float(w[treat_mask].sum() ** 2 / (w[treat_mask] ** 2).sum())
    info["ess_control"] = float(w[~treat_mask].sum() ** 2 / (w[~treat_mask] ** 2).sum())
    _require_four_group_support(df, marker, cancer_type, cohort, ps_model, info)
    return df, info


def _require_four_group_support(df: pl.DataFrame, marker, cancer_type, cohort, ps_model,
                                info: dict) -> None:
    """Refuse to draw a 2x2 panel that has an empty cell.

    A marker with no positives in an arm still plots: two of the four curves are
    simply absent, and the figure looks like a result rather than like nothing.
    That is the same silent-nothing failure the screen's own guards exist to
    prevent, so it raises here instead. Trimming can cause this even for a marker
    that passed the screen's support gates on the untrimmed frame.
    """
    counts = {}
    for m_val in (0, 1):
        for ici in (0, 1):
            sub = df.filter((pl.col("marker_value") == m_val) & (pl.col("PX_on_ICI") == ici))
            counts[(m_val, ici)] = (sub.height, int(sub["death"].sum()))
    info["group_counts"] = counts
    empty = [GROUP_STYLE[k]["label"] for k, (n, _) in counts.items() if n == 0]
    if empty:
        raise ValueError(
            f"{_hit_label(marker, cancer_type, cohort, ps_model)}: no patients in "
            f"{', '.join(empty)}"
            + (" after common-support trimming" if info.get("trimmed") else "")
            + ". A 2x2 KM cannot be drawn; the marker has no within-arm contrast here."
        )
    # Thin groups are recorded in `info` and surfaced by the export manifest's
    # n_marker_pos_ICI / d_marker_pos_ICI columns rather than printed per hit --
    # in a few-hundred-figure loop, a warning on every panel is noise.


def _check_recalibration_matches_screen(df: pl.DataFrame) -> None:
    """Assert the local recalibration reproduces `run_IPTW_analysis`'s, on real rows.

    The local copy exists only so this notebook can run without the screening
    dependencies. If the two ever diverge, every cancer-type KM would be drawn on
    a differently-trimmed population than the HR above it -- so when the real
    module is importable, prove they agree rather than assume it.
    """
    if not _SCREEN_IMPORTED:
        return
    mine = recalibrate_propensity_within_subset(df)
    theirs = _screen.recalibrate_propensity_within_subset(df)
    if not np.allclose(mine, theirs, rtol=1e-9, atol=1e-12):
        raise AssertionError(
            "recalibrate_propensity_within_subset in this notebook no longer matches "
            f"run_IPTW_analysis's (max abs diff {np.abs(mine - theirs).max():.3g}). "
            "Re-sync it before trusting any cancer-type KM."
        )


def effective_sample_size(weights: np.ndarray) -> float:
    """Kish ESS. Equals n when every weight is equal, so it is meaningful unweighted too."""
    weights = np.asarray(weights, dtype=float)
    denom = float((weights ** 2).sum())
    return float(weights.sum() ** 2 / denom) if denom > 0 else 0.0

## Rendering

`render_hit_figure` builds the two panels side by side — unweighted left, IPTW-weighted right —
over the same rows, and returns the figure for the exporter to save and close. Each panel's
legend carries the group's N, its event count, and (weighted) its ESS, because a curve resting
on 6 weighted patients and one resting on 200 look identical otherwise. The at-risk table is
drawn into the figure footer so each PNG stands on its own.

The stat under each panel is the **within-ICI marker log-rank** (marker+ vs marker− among ICI
patients) — the contrast the eye actually makes on a KM. It is *not* the interaction test the
hit was called on; the interaction HR from the Cox screen is printed in the title for that.
The weighted log-rank uses the weights as frequencies, so read its p-value as descriptive only.

In [ ]:
# lifelines warns on every weighted fit that non-integer weights make its naive
# variance estimate biased. That is true, it is why SHOW_CI defaults to False and
# why the weighted panels report ESS, and it is stated in the notebook's prose --
# but it fires four times per panel and buries the at-risk tables. Silence just
# that one warning; nothing else is suppressed.
warnings.filterwarnings(
    "ignore",
    message=".*weights are not integers.*",
    category=StatisticalWarning,
)


def _fit_group(ax, t, e, w, style, weighted: bool, show_ci: bool):
    """Fit and draw one of the four groups; returns its legend label or None if empty."""
    if len(t) == 0:
        return None
    kmf = KaplanMeierFitter()
    kwargs = dict(weights=w) if weighted else {}
    kmf.fit(t, e, label=style["label"], **kwargs)
    kmf.plot_survival_function(ax=ax, ci_show=show_ci, color=style["color"],
                               linestyle=style["linestyle"], linewidth=1.8)
    n, n_events = len(t), int(np.asarray(e).sum())
    if weighted:
        return f"{style['label']} (n={n}, ESS={effective_sample_size(w):.0f}, d={n_events})"
    return f"{style['label']} (n={n}, d={n_events})"


def _within_ici_logrank(frame: pl.DataFrame, weighted: bool):
    """Marker+ vs marker- log-rank among ICI-treated patients. None if a side is empty."""
    ici = frame.filter(pl.col("PX_on_ICI") == 1)
    pos = ici.filter(pl.col("marker_value") == 1)
    neg = ici.filter(pl.col("marker_value") == 0)
    if pos.is_empty() or neg.is_empty():
        return None
    try:
        res = logrank_test(
            pos["_time"].to_numpy(), neg["_time"].to_numpy(),
            event_observed_A=pos["death"].to_numpy(),
            event_observed_B=neg["death"].to_numpy(),
            weights_A=pos["IPTW_ATE"].to_numpy() if weighted else None,
            weights_B=neg["IPTW_ATE"].to_numpy() if weighted else None,
        )
        return float(res.p_value)
    except Exception as exc:  # a degenerate arm is a real outcome here, not a bug
        print(f"    log-rank ({'weighted' if weighted else 'unweighted'}) failed: {exc}")
        return None


def render_hit_figure(marker: str, cancer_type: str, cohort: str, ps_model: str,
                      hit_row: dict | None = None):
    """Build the two-panel figure for one hit and return (fig, info).

    Returns rather than shows: the batch export saves and closes each figure, and
    a few hundred open figures is a memory leak. Raises FileNotFoundError or
    ValueError if the population cannot be rebuilt -- `export_all_hits` records
    that as a skip.
    """
    frame, info = load_hit_population(marker, cancer_type, cohort, ps_model)
    frame = frame.with_columns((pl.col("tt_death") / TIME_DIVISOR).alias("_time"))

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), sharey=True)
    panel_p = {}

    for ax, weighted in zip(axes, (False, True)):
        labels = []
        for (m_val, ici), style in GROUP_STYLE.items():
            sub = frame.filter((pl.col("marker_value") == m_val) & (pl.col("PX_on_ICI") == ici))
            label = _fit_group(ax, sub["_time"].to_numpy(), sub["death"].to_numpy(),
                               sub["IPTW_ATE"].to_numpy(), style, weighted, SHOW_CI)
            if label:
                labels.append(label)
        panel_p[weighted] = _within_ici_logrank(frame, weighted)

        ax.set_title("IPTW-weighted (ATE)" if weighted else "Unweighted", fontsize=11)
        ax.set_xlabel(f"Time ({TIME_UNIT})")
        ax.set_ylim(0, 1.02)
        if MAX_FOLLOWUP is not None:
            ax.set_xlim(0, MAX_FOLLOWUP)
        ax.grid(alpha=0.25, linewidth=0.5)
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles, labels, fontsize=8, loc="upper right", frameon=False)
        p = panel_p[weighted]
        ax.text(0.02, 0.04, "marker+/- within ICI: " + ("log-rank p=n/a" if p is None else f"log-rank p={p:.3g}"),
                transform=ax.transAxes, fontsize=8, color="#444444")

    axes[0].set_ylabel("Survival probability")

    # Title: the Cox interaction estimate the hit was actually called on.
    bits = [f"{marker} — {cancer_type}", f"{cohort} / {ps_model}"]
    if hit_row:
        hr, p = hit_row.get("HR_first"), hit_row.get("min_p")
        est = []
        if hr is not None and np.isfinite(hr):
            est.append(f"HR(marker×ICI)={hr:.2f}")
        if p is not None and np.isfinite(p):
            est.append(f"p={p:.2e}")
        if hit_row.get("sig_in"):
            est.append(f"FDR-sig in: {hit_row['sig_in']}")
        if est:
            bits.append("  ".join(est))
        if hit_row.get("extreme_hr"):
            bits.append("!! extreme HR — possible model separation")
    pop = (f"n={info['n_analysis']} after common-support trim "
           f"(−{info['n_trimmed_out']} of {info['n_model_frame']})" if info["trimmed"]
           else f"n={info['n_analysis']} (UNTRIMMED — not the analysis population)")
    bits.append(pop)
    fig.suptitle("\n".join(bits), fontsize=11, y=1.0)
    fig.tight_layout(rect=(0, 0, 1, 0.90))

    if SHOW_AT_RISK:
        # Rendered into the PNG, not printed: an exported figure has to carry its
        # own at-risk numbers, or the thin-tail check is impossible from the file.
        _draw_at_risk(fig, frame)

    return fig, info


def _draw_at_risk(fig, frame: pl.DataFrame, n_bins: int = 6) -> None:
    """Write the at-risk table into the figure footer.

    A number-at-risk row is what separates a real tail separation from two
    patients, so it has to travel with the PNG rather than living in a notebook
    cell's stdout that no one exports.
    """
    upper = MAX_FOLLOWUP if MAX_FOLLOWUP is not None else float(frame["_time"].max())
    times = np.linspace(0, upper, n_bins)
    # The header and every data row are padded to the same label width, so the
    # counts line up under their time points in the monospace footer.
    label_w = 20
    lines = [f"{'At risk (' + TIME_UNIT + ')':<{label_w}}"
             + "".join(f"{t:>8.0f}" for t in times)]
    for (m_val, ici), style in GROUP_STYLE.items():
        sub = frame.filter((pl.col("marker_value") == m_val) & (pl.col("PX_on_ICI") == ici))
        t = sub["_time"].to_numpy()
        lines.append(f"{style['label']:<{label_w}}"
                     + "".join(f"{int((t >= x).sum()):>8d}" for x in times))
    fig.text(0.012, -0.02, "\n".join(lines), family="monospace", fontsize=7.5,
             va="top", ha="left", color="#333333")

## Export every hit

One PNG per (marker, cancer type, cohort, ps_model), each file holding the unweighted and
IPTW-weighted panels side by side. Files land in `KM_OUT_DIR`, one subdirectory per
cohort/ps_model specification so the four specs stay separable:

```
KM_curves/
  cohort1_covariates_only/KM_<marker>_<cancer_type>.png
  cohort1_covariates_plus_embeddings/...
  cohort2_covariates_only/...
  cohort2_covariates_plus_embeddings/...
  KM_export_manifest.csv
```

A hit whose population cannot be rebuilt — cancer type absent from that spec's IPTW dataset, or
one of the four groups emptied by trimming — is **skipped with its reason recorded**, not
drawn. The manifest carries one row per hit with the outcome, the group counts, the ESS, and
how many patients the trim removed, so the export is auditable without opening the PNGs and a
missing file is never a silent one.

Figures are closed as they are written rather than displayed, so the memory footprint stays flat
across a few hundred hits.

In [ ]:
# Lives under manuscript_figures/ beside the rendered panels, so a CLINICAL_FIGURES_OUT
# override moves it with them. KM_OUT_DIR overrides the whole path directly.
KM_OUT_DIR = os.environ.get(
    "KM_OUT_DIR",
    os.path.join(config.FIGURE_OUT_DIR, "figure5", "KM_curves"),
)
MANIFEST_NAME = "KM_export_manifest.csv"
DPI = 200
OVERWRITE = True   # False = leave existing PNGs alone (resume a partial export)

print(f"Writing to: {KM_OUT_DIR}")
print(f"{'' if OVERWRITE else 'Resume mode: existing PNGs will be kept.'}")

In [ ]:
def _safe_name(s: str) -> str:
    """Filename-safe token. Marker names carry '/' in a few fusion calls."""
    return "".join(ch if (ch.isalnum() or ch in "-_.") else "_" for ch in str(s))


def export_all_hits(panel_rows: pl.DataFrame, out_dir: str = None,
                    overwrite: bool = None) -> pl.DataFrame:
    """Render every hit to PNG; return the manifest.

    Never raises on a single bad hit -- an export of 200 figures that dies on
    number 3 is worse than one that records 3 failures and finishes. Only
    genuinely unexpected errors are re-raised.
    """
    out_dir = out_dir if out_dir is not None else KM_OUT_DIR
    overwrite = OVERWRITE if overwrite is None else overwrite
    os.makedirs(out_dir, exist_ok=True)

    records = []
    n_written = n_skipped = n_existing = 0

    for i, row in enumerate(panel_rows.iter_rows(named=True), start=1):
        marker, cancer = row["marker"], row["cancer_type"]
        cohort, ps_model = row["cohort"], row["ps_model"]
        spec = f"{cohort}_{ps_model}"
        spec_dir = os.path.join(out_dir, spec)
        fname = f"KM_{_safe_name(marker)}_{_safe_name(cancer)}.png"
        fpath = os.path.join(spec_dir, fname)

        rec = {
            "marker": marker, "cancer_type": cancer, "cohort": cohort,
            "ps_model": ps_model, "sig_in": row.get("sig_in"),
            "HR_markerxICI": row.get("HR_first"), "p_markerxICI": row.get("min_p"),
            "extreme_hr_flag": bool(row.get("extreme_hr", False)),
            "status": None, "reason": None, "file": None,
            "n_analysis": None, "n_trimmed_out": None,
            "ess_treated": None, "ess_control": None,
            "n_marker_pos_ICI": None, "n_marker_pos_nonICI": None,
            "n_marker_neg_ICI": None, "n_marker_neg_nonICI": None,
            "d_marker_pos_ICI": None, "d_marker_pos_nonICI": None,
        }

        if (not overwrite) and os.path.exists(fpath):
            rec.update(status="existing", file=os.path.relpath(fpath, out_dir))
            records.append(rec)
            n_existing += 1
            continue

        try:
            fig, info = render_hit_figure(marker, cancer, cohort, ps_model, hit_row=row)
        except (FileNotFoundError, ValueError) as exc:
            rec.update(status="skipped", reason=str(exc))
            records.append(rec)
            n_skipped += 1
            print(f"[{i:>4}/{panel_rows.height}] SKIP {spec}/{marker}/{cancer}: {exc}")
            continue

        os.makedirs(spec_dir, exist_ok=True)
        fig.savefig(fpath, dpi=DPI, bbox_inches="tight")
        plt.close(fig)   # batch export: never accumulate open figures

        counts = info.get("group_counts", {})
        rec.update(
            status="written", file=os.path.relpath(fpath, out_dir),
            n_analysis=info.get("n_analysis"), n_trimmed_out=info.get("n_trimmed_out"),
            ess_treated=info.get("ess_treated"), ess_control=info.get("ess_control"),
            n_marker_pos_ICI=counts.get((1, 1), (None, None))[0],
            n_marker_pos_nonICI=counts.get((1, 0), (None, None))[0],
            n_marker_neg_ICI=counts.get((0, 1), (None, None))[0],
            n_marker_neg_nonICI=counts.get((0, 0), (None, None))[0],
            d_marker_pos_ICI=counts.get((1, 1), (None, None))[1],
            d_marker_pos_nonICI=counts.get((1, 0), (None, None))[1],
        )
        records.append(rec)
        n_written += 1
        print(f"[{i:>4}/{panel_rows.height}] ok   {spec}/{fname}")

    manifest = pl.DataFrame(records, infer_schema_length=None)
    manifest.write_csv(os.path.join(out_dir, MANIFEST_NAME))

    print(f"\n{'=' * 70}")
    print(f"  written  {n_written}")
    if n_existing:
        print(f"  existing {n_existing} (resume mode)")
    print(f"  skipped  {n_skipped}")
    print(f"  manifest {os.path.join(out_dir, MANIFEST_NAME)}")
    print(f"{'=' * 70}")
    return manifest


if panels.is_empty():
    print("No significant hits to export.")
    manifest = pl.DataFrame()
else:
    print(f"Exporting {panels.height} hits to {KM_OUT_DIR}\n")
    manifest = export_all_hits(panels)

## Export summary

What was written, what was not, and why. A large `skipped` count is a finding about the hits
themselves — usually a marker with no positives left in one arm after common-support trimming —
not an export failure.

In [ ]:
if manifest.is_empty():
    print("Nothing exported.")
else:
    with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=100):
        print(manifest.group_by(["cohort", "ps_model", "status"])
                      .agg(pl.len().alias("n"))
                      .sort(["cohort", "ps_model", "status"]))

        skipped = manifest.filter(pl.col("status") == "skipped")
        if not skipped.is_empty():
            print(f"\n{skipped.height} hit(s) not drawn:")
            print(skipped.select(["marker", "cancer_type", "cohort", "ps_model", "reason"]))

        written = manifest.filter(pl.col("status") == "written")
        if not written.is_empty():
            # Thin marker+ cells are the usual reason a striking curve is not real,
            # so surface them here rather than leaving them to be found per-PNG.
            thin = written.filter((pl.col("n_marker_pos_ICI") < 10)
                                  | (pl.col("d_marker_pos_ICI") < 5))
            print(f"\n{written.height} figure(s) written; {thin.height} rest on a "
                  f"marker+/ICI cell under 10 patients or 5 deaths:")
            if not thin.is_empty():
                print(thin.select(["marker", "cancer_type", "cohort", "ps_model",
                                   "n_marker_pos_ICI", "d_marker_pos_ICI"])
                          .sort("n_marker_pos_ICI"))

## Reading these panels

- **The interaction is a ratio of ratios.** Two curves crossing does not make a hit, and four
  cleanly separated curves do not confirm one. What supports the reported HR is marker+ and
  marker− separating *differently* under ICI than under non-ICI.
- **Check the at-risk numbers before believing a tail.** The manifest's `n_marker_pos_ICI` /
  `d_marker_pos_ICI` columns are the fast way to do this across the whole export; separation
  that appears only past the point where that cell falls under ~10 is noise.
  `MIN_MARKER_POS_PER_ARM = 5` is the screen's floor, which is low.
- **Weighted vs unweighted is a confounding diagnostic.** If the two panels tell the same story,
  the interaction is not being carried by the propensity model. If weighting changes it
  materially, look at that spec's ESS in
  [01_pipeline.ipynb](01_pipeline.ipynb)'s diagnostics — a
  specification retaining under half its sample as effective size can move a curve on a handful
  of heavily-weighted patients. `ess_treated`/`ess_control` are in the manifest too.
- **`extreme_hr_flag` means separation, not a large effect.** A flagged panel usually shows one
  group with almost no events; the HR is a fitting artefact.
- **The log-rank under each panel is not the hit's test.** It is the within-ICI marker contrast,
  shown because it is the comparison the curves invite. The FDR-significant claim is the Cox
  interaction in the title.